# Problem Set 6 — Parts (c)–(f)

Returns are in percent per month. Each strategy is equal-weighted long the top three industries and short the bottom three. Ties follow the original Excel column order, consistent with part (a).


In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

DATA_PATH = "Problem_Set6_2026.xlsx"

ind = pd.read_excel(DATA_PATH, sheet_name="30 industries", header=3)
ind = ind.rename(columns={ind.columns[0]: "yyyymm"}).dropna(how="all")
ind["yyyymm"] = ind["yyyymm"].astype(int)
ind.index = pd.PeriodIndex(ind["yyyymm"].astype(str), freq="M", name="date")
R = ind.drop(columns="yyyymm").astype(float).mask(lambda x: x <= -999)
assert R.shape == (1200, 30) and R.notna().all().all()
assert R.index.is_unique and all((b - a).n == 1 for a, b in zip(R.index[:-1], R.index[1:]))

ff = pd.read_excel(DATA_PATH, sheet_name="Fama-French factors", header=2)
ff = ff.rename(columns={ff.columns[0]: "yyyymm"}).dropna(how="all")
ff["yyyymm"] = ff["yyyymm"].astype(int)
ff.index = pd.PeriodIndex(ff["yyyymm"].astype(str), freq="M", name="date")
ff = ff.drop(columns="yyyymm").astype(float).mask(lambda x: x <= -999)
assert ff.index.equals(R.index)
assert {"Mkt-RF", "SMB", "HML", "UMD"}.issubset(ff.columns)
print("Industry returns:", R.shape, R.index.min(), "to", R.index.max())
print("Factors:", ff.columns.tolist(), "| available UMD months:", ff["UMD"].notna().sum())

Industry returns: (1200, 30) 1926-07 to 2026-06
Factors: ['Mkt-RF', 'SMB', 'HML', 'RF', 'UMD'] | available UMD months: 1194


## (c)–(d) Industry momentum

- **12–1:** rank industries using compounded returns from t−12 through t−1, then earn their month-t returns.
- **12–1 skip:** rank using t−12 through t−2, skipping t−1, then earn month-t returns.

The 1–1 series is also computed for the factor regressions in (e) and (f), using the method in part (a).


In [2]:
def top_minus_bottom(formation_returns: pd.Series, realized_returns: pd.Series) -> float:
    top3 = formation_returns.nlargest(3, keep="first").index
    bottom3 = formation_returns.nsmallest(3, keep="first").index
    assert len(set(top3).intersection(bottom3)) == 0
    return float(realized_returns.loc[top3].mean() - realized_returns.loc[bottom3].mean())

rows = []
for i in range(1, len(R)):
    row = {"date": R.index[i], "1_1": top_minus_bottom(R.iloc[i - 1], R.iloc[i])}
    if i >= 12:
        c_signal = (1 + R.iloc[i - 12:i] / 100).prod(axis=0) - 1  # t-12 ... t-1
        d_signal = (1 + R.iloc[i - 12:i - 1] / 100).prod(axis=0) - 1  # t-12 ... t-2
        row["12_1"] = top_minus_bottom(c_signal, R.iloc[i])
        row["12_1_skip"] = top_minus_bottom(d_signal, R.iloc[i])
    rows.append(row)

mom = pd.DataFrame(rows).set_index("date").reindex(R.index)
assert mom["1_1"].count() == 1199
assert mom["12_1"].count() == mom["12_1_skip"].count() == 1188
assert mom["12_1"].first_valid_index() == pd.Period("1927-07", freq="M")

def perf_stats(x):
    x = x.dropna()
    mean, sd = x.mean(), x.std(ddof=1)
    return pd.Series({"T": len(x), "mean_%_mo": mean, "SD_%_mo": sd,
                      "t_mean": mean / (sd / np.sqrt(len(x))), "Sharpe_monthly": mean / sd})

performance = mom[["12_1", "12_1_skip"]].apply(perf_stats).T
print("(c) and (d):")
performance = performance.astype(float)
performance["T"] = performance["T"].astype(int)
display(performance.round(4))
print("Skip minus non-skip mean return (% points/month):",
      round(performance.loc["12_1_skip", "mean_%_mo"] - performance.loc["12_1", "mean_%_mo"], 6))

(c) and (d):


,T,mean_%_mo,SD_%_mo,t_mean,Sharpe_monthly
12_1,1188,0.8689,6.3265,4.7337,0.1373
12_1_skip,1188,0.8693,6.3114,4.7473,0.1377


Skip minus non-skip mean return (% points/month): 0.000426


## (e)–(f) Fama–French regressions

For each strategy, estimate:

\[
R^{W-L}_t = \alpha_3 + \beta_M (R_M-R_f)_t + \beta_S SMB_t + \beta_H HML_t + \epsilon_t
\]

\[
R^{W-L}_t = \alpha_4 + \beta_M (R_M-R_f)_t + \beta_S SMB_t + \beta_H HML_t + \beta_U UMD_t + \epsilon_t
\]

FF3 and FF4 use matching months for a fair comparison. The dependent variable is a zero-cost long–short return, so no additional risk-free rate is subtracted. Reported t-statistics use standard OLS errors.


In [3]:
regressors = ["Mkt-RF", "SMB", "HML", "UMD"]
reg_results = []
for strategy in ["1_1", "12_1", "12_1_skip"]:
    joined = pd.concat([mom[strategy].rename("WML"), ff[regressors]], axis=1).dropna()
    y = joined["WML"]
    for model, regcols in [("FF3", regressors[:3]), ("FF4", regressors)]:
        fit = sm.OLS(y, sm.add_constant(joined[regcols], has_constant="add")).fit()
        reg_results.append({"strategy": strategy, "model": model,
                            "start": str(joined.index[0]), "end": str(joined.index[-1]),
                            "T": len(joined), "alpha_%_mo": fit.params["const"],
                            "alpha_t": fit.tvalues["const"], "alpha_p": fit.pvalues["const"],
                            "UMD_beta": fit.params["UMD"] if model == "FF4" else np.nan,
                            "UMD_t": fit.tvalues["UMD"] if model == "FF4" else np.nan,
                            "R2": fit.rsquared})

regression_table = pd.DataFrame(reg_results)
assert regression_table["T"].tolist() == [1194, 1194, 1188, 1188, 1188, 1188]
assert np.all(regression_table.query("model == 'FF4'")["UMD_beta"] > 0)
print("(e) and (f):")
short_table = regression_table[["strategy", "model", "T", "alpha_%_mo", "alpha_t", "alpha_p", "UMD_beta"]].copy()
short_table["alpha_p"] = short_table["alpha_p"].map(lambda p: f"{p:.2g}")
display(short_table.round(4))



(e) and (f):


,strategy,model,T,alpha_%_mo,alpha_t,alpha_p,UMD_beta
0,1_1,FF3,1194,0.8001,4.9522,8.4e-07,NaN
1,1_1,FF4,1194,0.6564,3.9830,7.2e-05,0.1508
2,12_1,FF3,1188,1.1448,6.4681,1.4e-10,NaN
3,12_1,FF4,1188,0.2139,1.5868,0.11,0.9855
4,12_1_skip,FF3,1188,1.1170,6.2774,4.8e-10,NaN
5,12_1_skip,FF4,1188,0.1931,1.4104,0.16,0.9781


## Results and interpretation

**(c)** The 12–1 strategy earns **0.8689% per month** (t = 4.73, monthly Sharpe = 0.1373).

**(d)** Skipping the most recent month gives **0.8693% per month** (t = 4.75, Sharpe = 0.1377). The results are almost identical to (c), so skipping a month makes little difference in this sample.

**(e)** FF3 alphas are **0.8001%**, **1.1448%**, and **1.1170%** per month for 1–1, 12–1, and 12–1 skip, respectively. All are statistically significant, so FF3 does not fully price these strategies.

**(f)** Before adding UMD, I expected it to absorb much of the 12–1 momentum profit. That is what the regressions show: the 12–1 and skip alphas fall to **0.2139%** (t = 1.59) and **0.1931%** (t = 1.41), neither significant at 5%. The **1–1 alpha remains significant** at 0.6564% (t = 3.98). All three UMD loadings are positive, but the 1–1 loading is much smaller.

**AI check.** Adding UMD can make momentum alphas shrink because UMD is itself a momentum portfolio. This is useful for identifying a common factor, but it does not explain why the factor earns a premium. In particular, saying that FF4 prices *all* three strategies would be wrong here: 1–1 retains a significant alpha. These regressions cannot distinguish priced risk from investor under-reaction; that would require additional evidence about risk exposures or delayed price adjustment.

*Regression samples:* 1927-01–2026-06 for 1–1 (1,194 months) and 1927-07–2026-06 for the two 12–1 strategies (1,188 months). Both models use the same sample within each strategy so the change in alpha is not driven by different observations.
